In [1]:
import os
from pyspark.sql import SparkSession

# == CREDENTIALS ==
os.environ["AWS_ACCESS_KEY_ID"]     = "VOTRE_ACCESS_KEY"
os.environ["AWS_SECRET_ACCESS_KEY"] = "VOTRE_SECRET_KEY"
os.environ["AWS_SESSION_TOKEN"]     = "VOTRE_SESSION_TOKEN"

# Mémoire forcée à 6g 
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 6g pyspark-shell"

spark = (SparkSession.builder.appName("data-factory-ds")
    .config("spark.hadoop.fs.s3a.endpoint", "https://minio.lab.sspcloud.fr")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.session.token", os.environ["AWS_SESSION_TOKEN"])
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .master("local[2]")
    .getOrCreate())
print("Spark", spark.version, "démarré -",
      spark.sparkContext.getConf().get("spark.driver.memory"), "de RAM")

# Charger 5% du Silver et le garder en mémoire
df = spark.read.parquet("s3a://manar1305/data-factory-silver/accidents/").sample(fraction=0.05, seed=42).cache()
print("Données chargées :", df.count(), "lignes")

✓ Spark 4.1.1 démarré - 6g de RAM


✓ Données chargées : 386162 lignes


In [2]:
from pyspark.sql import functions as F

print("=== DISTRIBUTION DE LA GRAVITÉ (la cible) ===\n")
total = df.count()
dist = (df.groupBy("Severity").count()
          .withColumn("pourcentage", F.round(F.col("count")/total*100, 2))
          .orderBy("Severity"))
dist.show()

print("Observation clé : à quel point une seule classe domine-t-elle ?")

=== DISTRIBUTION DE LA GRAVITÉ (la cible) ===



+--------+------+-----------+
|Severity| count|pourcentage|
+--------+------+-----------+
|       1|  3447|       0.89|
|       2|307540|      79.64|
|       3| 64956|      16.82|
|       4| 10219|       2.65|
+--------+------+-----------+

→ Observation clé : à quel point une seule classe domine-t-elle ?


In [3]:
print("=== GRAVITÉ MOYENNE SELON L'HEURE DE LA JOURNÉE ===\n")
(df.withColumn("hour", F.hour("Start_Time"))
   .groupBy("hour")
   .agg(F.round(F.avg("Severity"), 3).alias("gravite_moyenne"),
        F.count("*").alias("nb_accidents"))
   .orderBy("hour")
   .show(24))

print("À quelles heures la gravité moyenne est-elle la plus élevée ?")

=== GRAVITÉ MOYENNE SELON L'HEURE DE LA JOURNÉE ===



+----+---------------+------------+
|hour|gravite_moyenne|nb_accidents|
+----+---------------+------------+
|   0|          2.225|        5610|
|   1|          2.186|        4835|
|   2|          2.206|        4790|
|   3|          2.208|        4259|
|   4|          2.236|        8025|
|   5|          2.227|       11285|
|   6|          2.214|       20334|
|   7|          2.189|       29162|
|   8|          2.187|       28871|
|   9|          2.229|       18215|
|  10|          2.219|       17118|
|  11|          2.209|       17852|
|  12|          2.215|       17756|
|  13|          2.208|       19920|
|  14|          2.203|       22491|
|  15|          2.203|       26287|
|  16|          2.203|       28868|
|  17|          2.208|       28836|
|  18|          2.227|       21624|
|  19|          2.247|       14886|
|  20|          2.247|       11295|
|  21|          2.238|        9282|
|  22|          2.243|        8259|
|  23|          2.193|        6302|
+----+---------------+------

In [4]:
print("=== GRAVITÉ MOYENNE SELON LA CONDITION MÉTÉO (top 15 des plus fréquentes) ===\n")
(df.groupBy("Weather_Condition")
   .agg(F.round(F.avg("Severity"), 3).alias("gravite_moyenne"),
        F.count("*").alias("nb_accidents"))
   .filter(F.col("nb_accidents") > 500)   # on ignore les météos trop rares
   .orderBy(F.desc("gravite_moyenne"))
   .show(15, truncate=False))

print("Les conditions dégradées (neige, pluie...) ont-elles une gravité plus élevée ?")

=== GRAVITÉ MOYENNE SELON LA CONDITION MÉTÉO (top 15 des plus fréquentes) ===



+-----------------------+---------------+------------+
|Weather_Condition      |gravite_moyenne|nb_accidents|
+-----------------------+---------------+------------+
|Overcast               |2.391          |19289       |
|Scattered Clouds       |2.382          |10199       |
|Clear                  |2.369          |40272       |
|Heavy Rain             |2.295          |1623        |
|Light Drizzle          |2.284          |1112        |
|Snow                   |2.257          |821         |
|Rain                   |2.255          |4177        |
|NULL                   |2.252          |8598        |
|Light Snow             |2.249          |6520        |
|Light Rain             |2.245          |17472       |
|Smoke                  |2.238          |680         |
|Light Rain with Thunder|2.223          |665         |
|Mostly Cloudy          |2.219          |50930       |
|Haze                   |2.219          |3822        |
|Partly Cloudy          |2.218          |34966       |
+---------

In [5]:
print("=== DISTANCE MOYENNE AFFECTÉE SELON LA GRAVITÉ ===\n")
(df.groupBy("Severity")
   .agg(F.round(F.avg("Distance(mi)"), 3).alias("distance_moy_miles"),
        F.round(F.max("Distance(mi)"), 1).alias("distance_max"))
   .orderBy("Severity")
   .show())

print("Les accidents graves bloquent-ils une plus longue distance de route ?")

=== DISTANCE MOYENNE AFFECTÉE SELON LA GRAVITÉ ===



+--------+------------------+------------+
|Severity|distance_moy_miles|distance_max|
+--------+------------------+------------+
|       1|             0.126|        11.5|
|       2|              0.56|       100.9|
|       3|             0.415|       138.3|
|       4|             1.431|        71.6|
+--------+------------------+------------+

→ Les accidents graves bloquent-ils une plus longue distance de route ?


In [6]:
from pyspark.sql import functions as F

# On repart du df brut pour tout reconstruire proprement
df_feat = df

# Renommer les colonnes à parenthèses ---
rename = {"Distance(mi)":"Distance_mi","Temperature(F)":"Temperature_F",
          "Humidity(%)":"Humidity_pct","Pressure(in)":"Pressure_in",
          "Visibility(mi)":"Visibility_mi","Wind_Speed(mph)":"Wind_Speed_mph"}
for old,new in rename.items():
    df_feat = df_feat.withColumnRenamed(old,new)

# Features temporelles de base ---
df_feat = (df_feat
    .withColumn("hour", F.hour("Start_Time"))
    .withColumn("dayofweek", F.dayofweek("Start_Time"))
    .withColumn("month", F.month("Start_Time"))
    .withColumn("is_weekend", F.dayofweek("Start_Time").isin(1,7).cast("int")))

# NOUVEAU : durée de l'accident (en minutes) ---
df_feat = df_feat.withColumn("duration_min",
    (F.col("End_Time").cast("long") - F.col("Start_Time").cast("long"))/60)

# NOUVEAU : saison (à partir du mois) ---
df_feat = df_feat.withColumn("saison",
    F.when(F.col("month").isin(12,1,2), "Hiver")
     .when(F.col("month").isin(3,4,5), "Printemps")
     .when(F.col("month").isin(6,7,8), "Ete")
     .otherwise("Automne"))

# NOUVEAU : heure de pointe (rush hour) ---
df_feat = df_feat.withColumn("is_rush_hour",
    (F.col("hour").isin(7,8,9,16,17,18)).cast("int"))

# Booléens route -> 0/1 ---
bool_cols = ["Amenity","Bump","Crossing","Give_Way","Junction","No_Exit",
             "Railway","Roundabout","Station","Stop","Traffic_Calming",
             "Traffic_Signal","Turning_Loop"]
for c in bool_cols:
    df_feat = df_feat.withColumn(c, F.col(c).cast("int"))

# Cible + poids de classe ---
df_feat = df_feat.withColumn("label", (F.col("Severity")-1).cast("double"))
counts = {r["Severity"]: r["count"] for r in df_feat.groupBy("Severity").count().collect()}
total,n = sum(counts.values()), len(counts)
weights = {k: total/(n*v) for k,v in counts.items()}
w_map = F.create_map([x for kv in weights.items() for x in (F.lit(kv[0]),F.lit(kv[1]))])
df_feat = df_feat.withColumn("classWeight", w_map[F.col("Severity")])

df_feat = df_feat.cache()
print("Features enrichies créées")
df_feat.select("Severity","duration_min","saison","is_rush_hour","is_weekend").show(5)

✓ Features enrichies créées


+--------+------------------+------+------------+----------+
|Severity|      duration_min|saison|is_rush_hour|is_weekend|
+--------+------------------+------+------------+----------+
|       3|168.06666666666666|   Ete|           0|         0|
|       2|106.13333333333334|   Ete|           0|         0|
|       3|              63.2|   Ete|           0|         0|
|       2|59.766666666666666|   Ete|           1|         0|
|       2| 57.43333333333333|   Ete|           0|         0|
+--------+------------------+------+------------+----------+
only showing top 5 rows


In [7]:
print("=== DURÉE MOYENNE DE L'ACCIDENT SELON LA GRAVITÉ ===\n")
(df_feat.groupBy("Severity")
   .agg(F.round(F.avg("duration_min"), 1).alias("duree_moy_min"))
   .orderBy("Severity")
   .show())

print("=== GRAVITÉ MOYENNE PAR SAISON ===\n")
(df_feat.groupBy("saison")
   .agg(F.round(F.avg("Severity"), 3).alias("gravite_moy"),
        F.count("*").alias("nb"))
   .orderBy(F.desc("gravite_moy"))
   .show())

=== DURÉE MOYENNE DE L'ACCIDENT SELON LA GRAVITÉ ===



+--------+-------------+
|Severity|duree_moy_min|
+--------+-------------+
|       1|         46.3|
|       2|        505.9|
|       3|         69.8|
|       4|       1219.1|
+--------+-------------+

=== GRAVITÉ MOYENNE PAR SAISON ===



+---------+-----------+------+
|   saison|gravite_moy|    nb|
+---------+-----------+------+
|      Ete|      2.238| 84517|
|Printemps|      2.222| 84594|
|  Automne|      2.209|103889|
|    Hiver|      2.189|113162|
+---------+-----------+------+



In [8]:
from pyspark.ml.feature import StringIndexer, Imputer, VectorAssembler

# Variables (SANS duration_min, qu'on a écartée)
numeric = ["Distance_mi","Temperature_F","Humidity_pct","Pressure_in",
           "Visibility_mi","Wind_Speed_mph","Start_Lat","Start_Lng"]
categorical = ["State","Sunrise_Sunset","saison"]   # +saison
temporal = ["hour","dayofweek","month","is_weekend","is_rush_hour"]  # +is_rush_hour

# Imputation des nulls numériques (médiane)
imp_out = [c+"_imp" for c in numeric]
imputer = Imputer(inputCols=numeric, outputCols=imp_out, strategy="median")

# Encodage des catégorielles
cat_idx = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
           for c in categorical]

# Assemblage final
feat = imp_out + bool_cols + temporal + [c+"_idx" for c in categorical]
assembler = VectorAssembler(inputCols=feat, outputCol="features", handleInvalid="skip")

# Découpage train/test (le MÊME pour les 3 modèles, pour comparer équitablement)
train, test = df_feat.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print("Transformations prêtes -", len(feat), "variables")
print("Train :", train.count(), "| Test :", test.count())

✓ Transformations prêtes - 29 variables


✓ Train : 309213 | Test : 76949


In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("=== MODÈLE 1 : RÉGRESSION LOGISTIQUE (baseline) ===\n")

lr = LogisticRegression(labelCol="label", featuresCol="features",
                        weightCol="classWeight", maxIter=20)

pipeline_lr = Pipeline(stages=[imputer]+cat_idx+[assembler, lr])

print("Entraînement... ")
model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

f1_lr  = MulticlassClassificationEvaluator(metricName="f1").evaluate(pred_lr)
acc_lr = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(pred_lr)
print(f"\n Régression Logistique → F1 : {f1_lr:.3f} | Accuracy : {acc_lr:.3f}")

=== MODÈLE 1 : RÉGRESSION LOGISTIQUE (baseline) ===

Entraînement... ⏳



✓ Régression Logistique → F1 : 0.445 | Accuracy : 0.355


In [10]:
from pyspark.ml.classification import RandomForestClassifier

print("=== MODÈLE 2 : RANDOM FOREST ===\n")

rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                            weightCol="classWeight",
                            numTrees=40, maxDepth=10, maxBins=64, seed=42)

pipeline_rf = Pipeline(stages=[imputer]+cat_idx+[assembler, rf])

print("Entraînement...")
model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

f1_rf  = MulticlassClassificationEvaluator(metricName="f1").evaluate(pred_rf)
acc_rf = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(pred_rf)
print(f"\n Random Forest → F1 : {f1_rf:.3f} | Accuracy : {acc_rf:.3f}")

=== MODÈLE 2 : RANDOM FOREST ===

Entraînement... ⏳ (2-3 min)



✓ Random Forest → F1 : 0.549 | Accuracy : 0.467


In [13]:
from pyspark.ml.classification import GBTClassifier, OneVsRest

print("=== MODÈLE 3 : GBT (Gradient Boosted Trees, via One-vs-Rest) ===\n")

gbt = GBTClassifier(labelCol="label", featuresCol="features",
                    maxIter=20, maxDepth=5, maxBins=64, seed=42)   # maxBins=64 ajouté
ovr = OneVsRest(classifier=gbt, labelCol="label", featuresCol="features")

pipeline_gbt = Pipeline(stages=[imputer]+cat_idx+[assembler, ovr])

print("Entraînement...")
model_gbt = pipeline_gbt.fit(train)
pred_gbt = model_gbt.transform(test)

f1_gbt  = MulticlassClassificationEvaluator(metricName="f1").evaluate(pred_gbt)
acc_gbt = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(pred_gbt)
print(f"\n GBT → F1 : {f1_gbt:.3f} | Accuracy : {acc_gbt:.3f}")

=== MODÈLE 3 : GBT (Gradient Boosted Trees, via One-vs-Rest) ===

Entraînement... ⏳ (3-5 min, le GBT est plus lent)



✓ GBT → F1 : 0.771 | Accuracy : 0.815


In [14]:
print("=== MATRICES DE CONFUSION DES 3 MODÈLES ===\n")
print("Rappel : 0=Sev1, 1=Sev2, 2=Sev3, 3=Sev4\n")

for nom, pred in [("Régression Logistique", pred_lr),
                  ("Random Forest", pred_rf),
                  ("GBT", pred_gbt)]:
    print(f"--- {nom} ---")
    pred.groupBy("label").pivot("prediction").count().orderBy("label").show()

=== MATRICES DE CONFUSION DES 3 MODÈLES ===

Rappel : 0=Sev1, 1=Sev2, 2=Sev3, 3=Sev4

--- Régression Logistique ---


+-----+-----+-----+-----+-----+
|label|  0.0|  1.0|  2.0|  3.0|
+-----+-----+-----+-----+-----+
|  0.0|  536|   27|   98|   37|
|  1.0|13278|20559|13933|13568|
|  2.0| 2855| 2187| 5212| 2694|
|  3.0|  270|  292|  401| 1002|
+-----+-----+-----+-----+-----+

--- Random Forest ---


+-----+----+-----+-----+-----+
|label| 0.0|  1.0|  2.0|  3.0|
+-----+----+-----+-----+-----+
|  0.0| 595|   24|   50|   29|
|  1.0|6757|25574|12244|16763|
|  2.0|1798| 1109| 8191| 1850|
|  3.0|  36|  276|   71| 1582|
+-----+----+-----+-----+-----+

--- GBT ---


+-----+----+-----+----+----+
|label| 0.0|  1.0| 2.0| 3.0|
+-----+----+-----+----+----+
|  0.0|NULL|  679|  19|NULL|
|  1.0|NULL|59713|1610|  15|
|  2.0|   3| 9916|3012|  17|
|  3.0|NULL| 1856|  87|  22|
+-----+----+-----+----+----+



In [15]:
# Le jeu de features ML enrichi, prêt pour la suite (Data Analyst / dashboard)
GOLD_FEATURES = "s3a://manar1305/data-factory-gold/ml_features/"

# On sélectionne les colonnes utiles (features + cible + identifiant)
features_finales = df_feat.select(
    "ID", "Severity", "label",
    "Distance_mi", "Temperature_F", "Humidity_pct", "Pressure_in",
    "Visibility_mi", "Wind_Speed_mph", "Start_Lat", "Start_Lng",
    "State", "Sunrise_Sunset", "saison",
    "hour", "dayofweek", "month", "is_weekend", "is_rush_hour",
    "Crossing", "Junction", "Traffic_Signal", "Stop", "Amenity", "Station"
)

print("Écriture des features en Gold...")
features_finales.write.mode("overwrite").parquet(GOLD_FEATURES)
print(" Features écrites :", GOLD_FEATURES)
print("  Nombre de lignes :", features_finales.count())

Écriture des features en Gold... ⏳


✓ Features écrites : s3a://manar1305/data-factory-gold/ml_features/
  Nombre de lignes : 386162


In [16]:
# On sauvegarde le modèle CHOISI (Random Forest pondéré) - réutilisable sans réentraînement
GOLD_MODEL = "s3a://manar1305/data-factory-gold/ml_model_rf/"

print("Sauvegarde du modèle... ⏳")
model_rf.write().overwrite().save(GOLD_MODEL)
print(" Modèle Random Forest sauvegardé :", GOLD_MODEL)

Sauvegarde du modèle... ⏳


✓ Modèle Random Forest sauvegardé : s3a://manar1305/data-factory-gold/ml_model_rf/
